# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zarnabrajpoot956-web/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane: Lane 4 — CTR / Engagement Opportunity Scoring.

Task type: Scoring / Ranking. This is not classification (I'm not sorting pages into fixed categories like "good" vs "bad") and not clustering (I'm not finding groups). I'm assigning each visible page a continuous score how far its CTR sits below what similar-positioned pages typically achieve and ranking pages by that score so a content editor can work through the list top-down.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no ground-truth label for "this page needs a
title rewrite" no one has manually tagged that. So I'm using a
proxy: ctr_gap = page's actual CTR median CTR of pages in the same
position_tier. A more negative gap means the page underperforms its
tier more severely. This is a proxy, not a direct measurement of
"needs review" a low CTR could have causes outside title/meta
(seasonality, wrong intent match, etc.), which I'll note as a
limitation, not treat as proven.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K (e.g., Precision@20) if I rank pages by
ctr_gap and take the top 20, what fraction are genuinely worth
reviewing? I'll approximate "genuinely worth reviewing" using the
same in-sample check I ran in Week 1 (below-median CTR at their tier,
with reasonable impression volume so the number isn't noise).
Alternative metric to consider: Spearman rank correlation between my
score and actual CTR shortfall, to check the ranking itself is sound,
not just the top-K cutoff.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
!git clone https://github.com/zarnabrajpoot956-web/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 39), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 10.40 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import glob

matches = glob.glob("**/content_refresh_anonymized.csv", recursive=True)
df = pd.read_csv(matches[0])

visible = df[df["impressions_90d"] >= 100].copy()

# Unit of analysis: one row = one page
# Compute the target/proxy: ctr_gap
tier_median = visible.groupby("position_tier")["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - tier_median

# Show the unit of analysis as a real dataframe
cols_to_show = ["position_tier", "content_type", "impressions_90d", "ctr", "ctr_gap"]
print(f"Unit of analysis: one row = one page. Shape: {visible[cols_to_show].shape}")
visible[cols_to_show].sort_values("ctr_gap").head(10)

Unit of analysis: one row = one page. Shape: (22006, 5)


,position_tier,content_type,impressions_90d,ctr,ctr_gap
12651,page_1,keyword article,155,0.0,-0.23
10144,page_1,keyword article,973,0.0,-0.23
20993,page_1,keyword article,144,0.0,-0.23
3751,page_1,keyword article,683,0.0,-0.23
10176,page_1,keyword article,106,0.0,-0.23
18973,page_1,keyword article,262,0.0,-0.23
17490,page_1,keyword article,678,0.0,-0.23
4854,page_1,keyword article,1258,0.0,-0.23
22356,page_1,keyword article,370,0.0,-0.23
22368,page_1,keyword article,118,0.0,-0.23


Notably, the 10 worst cases are all page_1 pages with CTR = 0.0 despite
meaningful impression volume (106–1,258) a #1-ranked page getting
zero clicks is a strong, explainable review candidate. All ten happen
to be "keyword article," but this may simply reflect that content type
being the most common overall rather than a unique weakness — worth
checking, not concluding, at this stage.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag any page with CTR below 0.10" fails because
CTR naturally varies by position tier 0.10 CTR is mediocre at
page_1 but excellent at "deep." A single global threshold either
misses all the top-tier underperformers or floods the queue with
normal deep-tier pages. My approach already requires a per-tier
comparison (grouping + relative scoring), which is exactly the kind
of conditional, data-driven computation that doesn't reduce to one
hand-written if-statement — and as I add more dimensions (content_type,
volume, freshness) as future refinements, a fixed rule would need an
unmanageable number of nested conditions, while a model/scoring
function handles it naturally.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.